In [64]:
# !pip install ibm_boto3 
# !pip uninstall kfp_components -y
# !pip install --no-cache-dir git+https://github.com/LukaszCmielowski/pipelines-components.git@rhoai_autorag_data_processing_pipeline

Found existing installation: kfp-components 1.11.0
Uninstalling kfp-components-1.11.0:
  Successfully uninstalled kfp-components-1.11.0
  Cloning https://github.com/LukaszCmielowski/pipelines-components.git (to revision rhoai_autorag_data_processing_pipeline) to /private/var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/pip-req-build-olgf90a5
  Running command git clone --filter=blob:none --quiet https://github.com/LukaszCmielowski/pipelines-components.git /private/var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/pip-req-build-olgf90a5
  Running command git checkout -b rhoai_autorag_data_processing_pipeline --track origin/rhoai_autorag_data_processing_pipeline
  Switched to a new branch 'rhoai_autorag_data_processing_pipeline'
  branch 'rhoai_autorag_data_processing_pipeline' set up to track 'origin/rhoai_autorag_data_processing_pipeline'.
  Resolved https://github.com/LukaszCmielowski/pipelines-components.git to commit 44d90b2cd8edd82278960b49be6c6ef1387b8510
  Installing build dependenc

In [5]:
import os
import json
import urllib.request

import ibm_boto3
from ibm_botocore.client import Config

In [6]:
# AWS_ACCESS_KEY_ID = "PLACE ACCESS KEY FOR YOUR S3 INSTANCE HERE"
# AWS_SECRET_ACCESS_KEY = "PLACE SECRET ACCESS KEY FOR YOUR S3 INSTANCE HERE"
# AWS_S3_ENDPOINT = "PLACE ENDPOINT URL FOR YOUR S3 INSTANCE HERE"
# AWS_DEFAULT_REGION = "PLACE REGION FOR YOUR S3 INSTANCE HERE"
# BUCKET_NAME = "BUCKET NAME TO UPLOAD DOCUMENTS"

os.environ["AWS_ACCESS_KEY_ID"] = "9df9c149900f45c5bb1e6520b37024fb"
os.environ["AWS_SECRET_ACCESS_KEY"] = "59591ea116a0b1a4919ca9314e7ecd4ba47bbcf559c01bf6"
os.environ["AWS_S3_ENDPOINT"] = "https://s3.us-south.cloud-object-storage.appdomain.cloud"
os.environ["AWS_DEFAULT_REGION"] = "us-south"
BUCKET_NAME = "autorag-dev-preview-dataset"

## Prepare experiment data

Initialize S3 client

In [7]:
s3_client = ibm_boto3.client(
    "s3",
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    config=Config(signature_version="s3v4"),
)

Upload documents

In [59]:
documents = [
    "https://www.ibm.com/downloads/documents/us-en/12bb2f913a3ba1a2",
    "https://www.ibm.com/downloads/documents/us-en/131cf8a39db327fd",
    "https://www.ibm.com/downloads/documents/us-en/131cf87ab633199f",
    "https://www.ibm.com/downloads/documents/us-en/1550f7eea8c0ded6",
    "https://www.ibm.com/downloads/documents/us-en/10a9980400afd114",
    "https://www.ibm.com/downloads/documents/us-en/10a9980468afdf4c",
    "https://www.ibm.com/downloads/documents/us-en/10a9980400afd11c",
    "https://www.ibm.com/downloads/documents/us-en/11ed3283ae56ec71"
]

for i, url in enumerate(documents):
    with urllib.request.urlopen(url) as response:
        content = response.read()
    s3_client.put_object(Bucket=BUCKET_NAME, Key=f"document_{i}.pdf", Body=content)

Upload benchmark dataset

In [60]:
benchmark = [
    {
        "question": "What was IBM's revenue in the first quarter of 2024?",
        "correct_answers": [
            "Revenue of $14.5 billion, up 1 percent, up 3 percent at constant currency."
        ],
        "correct_answer_document_ids": [
            "ibm-1q24-earnings-press-release.pdf"
        ]
    },
    {
        "question": "What did IBM announce regarding HashiCorp in first quarter 2024?",
        "correct_answers": [
            "IBM announced its intent to acquire HashiCorp, Inc. for $35 per share in cash, representing an enterprise value of $6.4 billion. The transaction was expected to close by the end of 2024."
        ],
        "correct_answer_document_ids": [
            "ibm-1q24-earnings-press-release.pdf"
        ]
    }
]

res = s3_client.put_object(Bucket=BUCKET_NAME, Key="benchmark.json", Body=json.dumps(benchmark))

Look up bucket contents

In [8]:
s3_client.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix="",
).get("Contents", [])

[{'Key': 'benchmark.json',
  'LastModified': datetime.datetime(2026, 2, 19, 13, 38, 58, 841000, tzinfo=tzutc()),
  'ETag': '"ed5d5d70cd5f6f4f229f9242687cf152"',
  'ChecksumAlgorithm': ['CRC32'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 606,
  'StorageClass': 'STANDARD'},
 {'Key': 'document_0.pdf',
  'LastModified': datetime.datetime(2026, 2, 19, 13, 38, 49, 421000, tzinfo=tzutc()),
  'ETag': '"602598e51a6c8e06a7c80d26c4aee300"',
  'ChecksumAlgorithm': ['CRC32'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 230841,
  'StorageClass': 'STANDARD'},
 {'Key': 'document_1.pdf',
  'LastModified': datetime.datetime(2026, 2, 19, 13, 38, 50, 655000, tzinfo=tzutc()),
  'ETag': '"64c5beddfe7c50bb6b827d7f6d825118"',
  'ChecksumAlgorithm': ['CRC32'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 280313,
  'StorageClass': 'STANDARD'},
 {'Key': 'document_2.pdf',
  'LastModified': datetime.datetime(2026, 2, 19, 13, 38, 52, 89000, tzinfo=tzutc()),
  'ETag': '"9d0309206828a5c38e2e99aa036daa6d"',
  'ChecksumA

## Process input documents

In [9]:
from kfp import local
from kfp_components.pipelines.data_processing.autorag.pipeline import data_processing_pipeline

local.init(
    runner=local.SubprocessRunner(use_venv=False),
    pipeline_root="./local_outputs",
    raise_on_error=True,
)

result = data_processing_pipeline(
    test_data_secret_name="autorag-input-data-secret",
    input_data_secret_name="autorag-input-data-secret",
    test_data_bucket_name="autorag-dev-preview-dataset",
    test_data_key="benchmark.json",
    input_data_bucket_name="autorag-dev-preview-dataset",
    input_data_key="",
    sampling_config={"max_size_gigabytes": 1},
)

15:03:37.672 - INFO - Running pipeline: 'autorag-data-processing-pipeline'
--------------------------------------------------------------------------------
15:03:37.674 - INFO - Executing task 'test-data-loader'
15:03:37.674 - INFO - Streamed logs:



/Users/wnowogor/PycharmProjects/ai4rag/.venv/lib/python3.14/site-packages/kfp/local/subprocess_task_handler.py:66: RuntimeWarning: You may be attemping to run a task that uses custom or non-Python base image 'wnowogorski-org/autorag_data_loading' in a Python environment. This may result in incorrect dependencies and/or incorrect behavior. Consider using the 'DockerRunner' to run this task in a container.
  warnings.warn(


    
    [notice] A new release of pip is available: 24.3.1 -> 26.0.1
    [notice] To update, run: pip install --upgrade pip
    [KFP Executor 2026-02-19 15:03:38,234 INFO]: Looking for component `test_data_loader` in --component_module_path `/var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.rH2fVt2TDX/ephemeral_component.py`
    [KFP Executor 2026-02-19 15:03:38,234 INFO]: Loading KFP component "test_data_loader" from /var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.rH2fVt2TDX/ephemeral_component.py (directory "/var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.rH2fVt2TDX" and module name "ephemeral_component")
    [KFP Executor 2026-02-19 15:03:38,235 INFO]: Got executor_input:
    {
        "inputs": {
            "parameterValues": {
                "test_data_path": "benchmark.json",
                "test_data_bucket_name": "autorag-dev-preview-dataset"
            }
        },
        "outputs": {
            "artifacts": {
                "test_data": {
                

/Users/wnowogor/PycharmProjects/ai4rag/.venv/lib/python3.14/site-packages/kfp/local/subprocess_task_handler.py:66: RuntimeWarning: You may be attemping to run a task that uses custom or non-Python base image 'wnowogorski-org/autorag_data_loading' in a Python environment. This may result in incorrect dependencies and/or incorrect behavior. Consider using the 'DockerRunner' to run this task in a container.
  warnings.warn(


    
    [notice] A new release of pip is available: 24.3.1 -> 26.0.1
    [notice] To update, run: pip install --upgrade pip
    [KFP Executor 2026-02-19 15:03:40,325 INFO]: Looking for component `documents_sampling` in --component_module_path `/var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.W6i7EO2wbA/ephemeral_component.py`
    [KFP Executor 2026-02-19 15:03:40,325 INFO]: Loading KFP component "documents_sampling" from /var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.W6i7EO2wbA/ephemeral_component.py (directory "/var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.W6i7EO2wbA" and module name "ephemeral_component")
    [KFP Executor 2026-02-19 15:03:40,326 INFO]: Got executor_input:
    {
        "inputs": {
            "artifacts": {
                "test_data": {
                    "artifacts": [
                        {
                            "name": "test_data",
                            "type": {
                                "schemaTitle": "system.Artifact",


/Users/wnowogor/PycharmProjects/ai4rag/.venv/lib/python3.14/site-packages/kfp/local/subprocess_task_handler.py:66: RuntimeWarning: You may be attemping to run a task that uses custom or non-Python base image 'wnowogorski-org/autorag_data_loading' in a Python environment. This may result in incorrect dependencies and/or incorrect behavior. Consider using the 'DockerRunner' to run this task in a container.
  warnings.warn(


    
    [notice] A new release of pip is available: 24.3.1 -> 26.0.1
    [notice] To update, run: pip install --upgrade pip
    [KFP Executor 2026-02-19 15:03:42,114 INFO]: Looking for component `text_extraction` in --component_module_path `/var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.inWyJKRLGF/ephemeral_component.py`
    [KFP Executor 2026-02-19 15:03:42,114 INFO]: Loading KFP component "text_extraction" from /var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.inWyJKRLGF/ephemeral_component.py (directory "/var/folders/3x/80z00mdn0kl0r08j0zc2ssr40000gn/T/tmp.inWyJKRLGF" and module name "ephemeral_component")
    [KFP Executor 2026-02-19 15:03:42,115 INFO]: Got executor_input:
    {
        "inputs": {
            "artifacts": {
                "sampled_documents_descriptor": {
                    "artifacts": [
                        {
                            "name": "sampled_documents",
                            "type": {
                                "schemaTitle"

Look up sampling configuraton

In [11]:
result

AttributeError: 'PipelineTask' object has no attribute 'documents_sampling_task'

## Run ai4rag experiment

In [52]:
!pip install docker


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
